# Semantic Cache: Cost-Efficient Short Term Memory

**Semantic Cache** is a form of short-term memory that stores previous LLM responses and serves them for semantically equivalent future queries — without calling the LLM again.

Unlike an exact-match cache (which only hits on identical strings), a semantic cache uses vector similarity to recognise that *"I forgot my password"* and *"Can't remember my login"* are asking the same thing.

```
User query → embed (VoyageAI) → $vectorSearch (MongoDB)
                                        ↓
               similarity ≥ threshold? → CACHE HIT  → return stored response
               similarity < threshold? → CACHE MISS → call LLM → store response
```

Benefits:
- **Cost reduction** — repeated or similar questions skip the LLM entirely
- **Latency reduction** — cache hits return in milliseconds instead of seconds
- **Consistency** — semantically identical questions always get the same answer

> **Scenario:** A travel FAQ assistant. Questions like *"Is parking included?"* and *"Do any listings have parking?"* should hit the same cached answer. Novel questions go to the LLM and are cached for future reuse.

In [ ]:
import { MongoClient } from 'mongodb';
import { ChatAnthropic } from '@langchain/anthropic';

// ← Set VOYAGE_API_KEY as a Codespace secret, or paste it here as a fallback
const VOYAGE_API_KEY = process.env.VOYAGE_API_KEY ?? 'pa-...';

// ← Set ANTHROPIC_API_KEY as a Codespace secret, or paste it here as a fallback
const ANTHROPIC_API_KEY = process.env.ANTHROPIC_API_KEY ?? 'sk-ant-...';

const client = new MongoClient(process.env.MONGODB_URI!, { appName: 'devrel-workshop-agentmemory-langchain' });
await client.connect();
const db    = client.db('agent_memory_lab');
const cache = db.collection('semantic_cache');

console.log('Connected.');

## Cache document schema

Each cache entry stores the canonical question, the LLM-generated answer, the VoyageAI embedding for similarity search, and usage statistics (hit count, TTL).

In [ ]:
interface CacheEntry {
  question:  string;
  answer:    string;
  embedding: number[];
  hits:      number;
  createdAt: Date;
  expiresAt: Date;
}

const CACHE_TTL_SECONDS    = 3600;
const SIMILARITY_THRESHOLD = 0.88;

console.log(`Cache TTL: ${CACHE_TTL_SECONDS}s | Similarity threshold: ${SIMILARITY_THRESHOLD}`);

## Embedding helper

In [ ]:
async function embed(texts: string[], model: string, inputType: 'document' | 'query'): Promise<number[][]> {
  const res = await fetch('https://api.voyageai.com/v1/embeddings', {
    method: 'POST',
    headers: { 'Content-Type': 'application/json', 'Authorization': `Bearer ${VOYAGE_API_KEY}` },
    body: JSON.stringify({ input: texts, model, input_type: inputType }),
  });
  if (!res.ok) throw new Error(await res.text());
  const json = await res.json() as { data: { embedding: number[] }[] };
  return json.data.map(d => d.embedding);
}

console.log('Embedding helper ready.');

## Creating the vector search index on the cache

The `$vectorSearch` index on `embedding` enables sub-millisecond semantic similarity lookups. A `filter` field on `expiresAt` lets the aggregation skip expired entries without a separate query.

In [ ]:
const CACHE_INDEX = 'semantic_cache_index';

try {
  await cache.dropSearchIndex(CACHE_INDEX);
  await new Promise(r => setTimeout(r, 2000));
} catch { /* index did not exist */ }

await cache.createSearchIndex({
  name: CACHE_INDEX,
  type: 'vectorSearch',
  definition: {
    fields: [
      { type: 'vector', path: 'embedding', numDimensions: 1024, similarity: 'cosine' },
      { type: 'filter', path: 'expiresAt' },
    ],
  },
});

console.log('Waiting for cache index to be READY...');
for (let i = 0; i < 30; i++) {
  await new Promise(r => setTimeout(r, 2000));
  const [idx] = await cache.listSearchIndexes(CACHE_INDEX).toArray();
  console.log(' status:', idx?.status);
  if (idx?.status === 'READY') break;
}

## The semantic cache lookup

On every query, we embed the question with `voyage-4-lite` and run `$vectorSearch`. If the top result exceeds the similarity threshold, it is a cache hit. The entry's hit counter is incremented and the stored answer is returned immediately — no LLM call.

In [ ]:
async function cacheGet(question: string): Promise<string | null> {
  const [qVec] = await embed([question], 'voyage-4-lite', 'query');
  const now = new Date();

  const results = await cache.aggregate([
    {
      $vectorSearch: {
        index:         CACHE_INDEX,
        path:          'embedding',
        queryVector:   qVec,
        numCandidates: 20,
        limit:         1,
        filter:        { expiresAt: { $gt: now } },
      },
    },
    {
      $project: {
        _id:      1,
        question: 1,
        answer:   1,
        hits:     1,
        score:    { $meta: 'vectorSearchScore' },
      },
    },
  ]).toArray();

  if (results.length === 0) return null;

  const top = results[0] as any;
  if (top.score < SIMILARITY_THRESHOLD) return null;

  await cache.updateOne({ _id: top._id }, { $inc: { hits: 1 } });
  console.log(`CACHE HIT  (score: ${top.score.toFixed(3)}, hits: ${top.hits + 1}) — "${top.question}"`);
  return top.answer as string;
}

console.log('cacheGet ready.');

## The cache store

On a cache miss, the LLM generates an answer. The question is embedded with `voyage-4-large` and the full entry is stored in MongoDB for future lookups.

In [ ]:
const llm = new ChatAnthropic({ model: 'claude-haiku-4-5-20251001', apiKey: ANTHROPIC_API_KEY });

async function cacheSet(question: string, answer: string): Promise<void> {
  const [embedding] = await embed([question], 'voyage-4-large', 'document');
  const now = new Date();

  const entry: CacheEntry = {
    question,
    answer,
    embedding,
    hits:      0,
    createdAt: now,
    expiresAt: new Date(now.getTime() + CACHE_TTL_SECONDS * 1000),
  };

  await cache.insertOne(entry);
  console.log(`Stored in cache: "${question.slice(0, 60)}"`);
}

async function ask(question: string): Promise<string> {
  const cached = await cacheGet(question);
  if (cached) return cached;

  console.log('CACHE MISS — calling LLM...');
  const start = Date.now();
  const response = await llm.invoke(
    'You are a concise travel FAQ assistant for an Airbnb-style platform. ' +
    'Answer in 2-3 sentences.\n\n' + question
  );
  console.log(`LLM responded in ${Date.now() - start}ms`);

  const answer = response.content as string;
  await cacheSet(question, answer);
  return answer;
}

console.log('ask() ready.');

## Seeding the cache with common questions

Pre-warm the cache with FAQ answers so the similarity lookups have entries to match against.

In [ ]:
const faqs = [
  'How do I reset my password on the platform?',
  'What is the cancellation policy for bookings?',
  'Do listings include parking?',
  'Is WiFi available in all properties?',
  'Can I bring my pet to the listing?',
];

for (const q of faqs) {
  await ask(q);
}

console.log(`\nCache seeded with ${faqs.length} entries.`);

## Cache hit — semantic variants skip the LLM

These questions are worded differently from the cached entries but are semantically equivalent. They should all hit the cache and return instantly.

In [ ]:
const variants = [
  "Can't remember my login credentials — how do I recover access?",
  'What happens if I need to cancel my reservation?',
  'Is there parking available at the properties?',
];

for (const q of variants) {
  console.log(`\nQuery: "${q}"`);
  const answer = await ask(q);
  console.log('Answer:', answer.slice(0, 120) + '...');
}

## Cache miss — novel questions call the LLM

A genuinely different question falls below the similarity threshold and must be answered by the LLM. The new answer is stored for future reuse.

In [ ]:
const novelQuestion = 'What are the check-in hours for most listings?';

console.log(`Query: "${novelQuestion}"`);
const novelAnswer = await ask(novelQuestion);
console.log('Answer:', novelAnswer);

## Inspecting cache statistics

Hit counts per entry show which questions are asked most often. High-hit entries are the most valuable to keep in the cache.

In [ ]:
const entries = await cache
  .find({}, { projection: { question: 1, hits: 1, expiresAt: 1 } })
  .sort({ hits: -1 })
  .toArray();

console.log('Cache entries by hit count:\n');
entries.forEach((e: any) => {
  const ttlSec = Math.round((new Date(e.expiresAt).getTime() - Date.now()) / 1000);
  console.log(`  [hits: ${e.hits}] "${(e.question as string).slice(0, 60)}" (TTL: ${ttlSec}s)`);
});

const totalHits = entries.reduce((s: number, e: any) => s + e.hits, 0);
console.log(`\nTotal entries: ${entries.length} | Total hits served from cache: ${totalHits}`);

## Below-threshold query — a completely different topic

A question about a completely different subject should not match any cached travel FAQ and must go to the LLM.

In [ ]:
const offTopicQuestion = 'What is the capital of France?';
console.log(`Query: "${offTopicQuestion}"`);

const [qVec] = await embed([offTopicQuestion], 'voyage-4-lite', 'query');
const topMatch = await cache.aggregate([
  {
    $vectorSearch: {
      index: CACHE_INDEX, path: 'embedding', queryVector: qVec,
      numCandidates: 20, limit: 1,
    },
  },
  { $project: { question: 1, score: { $meta: 'vectorSearchScore' } } },
]).toArray();

const bestScore = topMatch.length > 0 ? (topMatch[0] as any).score : 0;
console.log(`Best match score: ${bestScore.toFixed(3)} (threshold: ${SIMILARITY_THRESHOLD})`);
console.log(bestScore < SIMILARITY_THRESHOLD
  ? '→ Below threshold — CACHE MISS as expected.'
  : '→ Above threshold — CACHE HIT.');

In [ ]:
await cache.deleteMany({});
try { await cache.dropSearchIndex(CACHE_INDEX); } catch {}
await client.close();
console.log('Done.');